# Final Project — Infectious Cases (PostgreSQL)
**Relational Databases for DS · goit-rdb-fp**

⚠️ Для запуску потрібен оригінальний `infectious_cases.csv`. Цей notebook **не підміняє** реальні дані синтетичними. Запускайте клітинки послідовно в Google Colab. Скриншоти отримуйте з результатів після виконання.

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 49.7 MB/s eta 0:00:00


In [2]:
import os, re, io
from pathlib import Path
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text
from IPython.display import display
pg=pgserver.get_server('/tmp/fp_pg',cleanup_mode='stop')
engine=create_engine(pg.get_uri(),future=True)
def run(sql, params=None):
    with engine.begin() as c: return c.execute(text(sql),params or {})
def query(sql,params=None):
    with engine.connect() as c: return pd.read_sql_query(text(sql),c,params=params)
def ident(s): return '"'+str(s).replace('"','""')+'"'
print(query('SELECT version() AS postgres_version').iloc[0,0])

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


## Task 1 — Schema and CSV import
Виберіть оригінальний CSV через діалог завантаження. Колонки збережені в початковому вигляді (усі як TEXT), щоб коректно обробляти порожні рядки та інші нечислові значення.

In [3]:
run('CREATE SCHEMA IF NOT EXISTS pandemic')
run('SET search_path TO pandemic, public')  # SQL requirement; per-connection setting is also set below
# Recreate engine so new connections use the default search_path.
engine.dispose()
engine=create_engine(pg.get_uri(),future=True,connect_args={'options':'-csearch_path=pandemic,public'})
print(query('SHOW search_path'))

       search_path
0  pandemic,public


In [5]:
if os.path.exists('/content'):
    from google.colab import files
    uploaded=files.upload()
    if not uploaded: raise RuntimeError('Завантажте infectious_cases.csv')
    csv_path=next((name for name in uploaded if name.lower().endswith('.csv')),None)
    if csv_path is None: raise ValueError('Потрібен файл CSV')
    raw_bytes=uploaded[csv_path]
else:
    csv_path='infectious_cases.csv'
    if not os.path.exists(csv_path): raise FileNotFoundError('Покладіть infectious_cases.csv у поточну папку')
    raw_bytes=Path(csv_path).read_bytes()
try:
    df=pd.read_csv(io.BytesIO(raw_bytes),dtype=str,keep_default_na=False,encoding='utf-8-sig',sep=None,engine='python')
except UnicodeDecodeError:
    df=pd.read_csv(io.BytesIO(raw_bytes),dtype=str,keep_default_na=False,encoding='cp1251',sep=None,engine='python')
df.columns=[str(x).strip().lstrip('\ufeff') for x in df.columns]
assert {'Entity','Code','Year','Number_rabies'}.issubset(df.columns),f'Відсутні колонки: {set(["Entity","Code","Year","Number_rabies"])-set(df.columns)}'
assert len(df)>0,'CSV порожній'
assert not df.columns.duplicated().any(),'Повторювані назви колонок'
df.to_sql('infectious_cases',engine,schema='pandemic',if_exists='replace',index=False,chunksize=1000,dtype={col:__import__('sqlalchemy').Text() for col in df.columns})
print('Source rows:',len(df),'Columns:',len(df.columns))
display(query('SELECT * FROM pandemic.infectious_cases LIMIT 10'))

Saving infectious_cases.csv to infectious_cases.csv
Source rows: 10521 Columns: 12


,Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
0,Afghanistan,AFG,1980,,6160,0,,,,,,
1,Afghanistan,AFG,1981,,5859,0,,,,,,
2,Afghanistan,AFG,1982,,9730,0,,,,,,
3,Afghanistan,AFG,1983,,13937,0,,,,,,
4,Afghanistan,AFG,1984,,3864,0,,,,,,
5,Afghanistan,AFG,1985,,13867,0,,,,,,
6,Afghanistan,AFG,1986,,12901,0,,,,,,
7,Afghanistan,AFG,1987,,4396,0,,,,,,
8,Afghanistan,AFG,1988,,2149,0,,,,,,
9,Afghanistan,AFG,1989,,385,0,,,,,,


## Task 2 — 3NF normalization
`entities`: унікальна пара `(Entity, Code)`; `disease_metrics`: окремі спостереження, пов’язані зовнішнім ключем `entity_id`. Збережено всі інші поля вихідного CSV.

In [6]:
run('DROP TABLE IF EXISTS pandemic.disease_metrics CASCADE')
run('DROP TABLE IF EXISTS pandemic.entities CASCADE')
run("""CREATE TABLE pandemic.entities (
  id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  entity TEXT NOT NULL,
  code TEXT NOT NULL,
  CONSTRAINT uq_entity_code UNIQUE (entity, code)
)""")
run("""INSERT INTO pandemic.entities (entity, code)
 SELECT DISTINCT "Entity", "Code" FROM pandemic.infectious_cases""")
other_cols=[x for x in df.columns if x not in ('Entity','Code')]
assert 'Year' in other_cols and 'Number_rabies' in other_cols
cols_ddl=',\n  '.join(ident(c)+' TEXT' for c in other_cols)
run('CREATE TABLE pandemic.disease_metrics (id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY, entity_id BIGINT NOT NULL REFERENCES pandemic.entities(id), '+cols_ddl+')')
select_cols=', '.join('s.'+ident(c) for c in other_cols)
insert_cols=', '.join(ident(c) for c in other_cols)
run('INSERT INTO pandemic.disease_metrics (entity_id, '+insert_cols+') SELECT e.id, '+select_cols+' FROM pandemic.infectious_cases s JOIN pandemic.entities e ON e.entity=s."Entity" AND e.code=s."Code"')
display(query('SELECT * FROM pandemic.entities LIMIT 10'))
display(query('SELECT * FROM pandemic.disease_metrics LIMIT 10'))
print('Source/normalized counts:')
display(query('SELECT (SELECT COUNT(*) FROM pandemic.infectious_cases) AS source_rows, (SELECT COUNT(*) FROM pandemic.disease_metrics) AS normalized_rows, (SELECT COUNT(*) FROM pandemic.entities) AS unique_entities'))

,id,entity,code
0,1,Morocco,MAR
1,2,Iran,IRN
2,3,Russia,RUS
3,4,Democratic Republic of Congo,COD
4,5,North America (WB),
5,6,Ethiopia,ETH
6,7,Trinidad and Tobago,TTO
7,8,Israel,ISR
8,9,Antigua and Barbuda,ATG
9,10,Libya,LBY


,id,entity_id,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
0,1,143,1980,,6160,0,,,,,,
1,2,143,1981,,5859,0,,,,,,
2,3,143,1982,,9730,0,,,,,,
3,4,143,1983,,13937,0,,,,,,
4,5,143,1984,,3864,0,,,,,,
5,6,143,1985,,13867,0,,,,,,
6,7,143,1986,,12901,0,,,,,,
7,8,143,1987,,4396,0,,,,,,
8,9,143,1988,,2149,0,,,,,,
9,10,143,1989,,385,0,,,,,,


Source/normalized counts:


,source_rows,normalized_rows,unique_entities
0,10521,10521,245


## Task 3 — Rabies summary by entity
Порожні/нечислові значення не беруть участі в агрегуванні. Сортування AVG DESC, лише 10 рядків.

In [7]:
sql_task3 = '''SELECT e.id AS entity_id, e.entity, e.code,
  ROUND(AVG(NULLIF(TRIM(m."Number_rabies"), '')::NUMERIC), 3) AS avg_rabies,
  MIN(NULLIF(TRIM(m."Number_rabies"), '')::NUMERIC) AS min_rabies,
  MAX(NULLIF(TRIM(m."Number_rabies"), '')::NUMERIC) AS max_rabies,
  SUM(NULLIF(TRIM(m."Number_rabies"), '')::NUMERIC) AS sum_rabies
FROM pandemic.disease_metrics m
JOIN pandemic.entities e ON e.id=m.entity_id
WHERE TRIM(m."Number_rabies") ~ '^[+-]?[0-9]+([.][0-9]+)?$'
GROUP BY e.id,e.entity,e.code
ORDER BY avg_rabies DESC NULLS LAST
LIMIT 10'''
display(query(sql_task3))

,entity_id,entity,code,avg_rabies,min_rabies,max_rabies,sum_rabies
0,102,World,OWID_WRL,20192.370,14075.5080,24744.6580,605771.1110
1,170,Lower Middle Income (WB),,15193.959,10202.5280,19182.7950,455818.7815
2,193,South Asia (WB),,11729.889,7271.2754,15361.8780,351896.6626
3,19,South-East Asia Region (WHO),,11424.327,6806.0127,15641.9580,342729.8068
4,187,G20,,10189.046,6339.0796,13164.8810,305671.3801
5,182,India,IND,8599.173,5425.8726,11121.1420,257975.1984
6,33,Sub-Saharan Africa (WB),,4748.268,4356.8184,5017.9770,142448.0427
7,232,African Region (WHO),,4649.702,4283.0730,4917.7840,139491.0713
8,130,Low Income (WB),,3568.105,3014.6170,3873.5330,107043.1635
9,241,East Asia & Pacific (WB),,3427.461,2009.9395,4591.0107,102823.8340


## Task 4 — January 1, current date, difference in completed years
`AGE` + `DATE_PART` рахує **повні** роки, а не лише різницю номерів років.

In [8]:
sql_task4 = '''SELECT m.entity_id, m."Year" AS source_year,
 MAKE_DATE(TRIM(m."Year")::INTEGER,1,1) AS year_start,
 CURRENT_DATE AS today,
 DATE_PART('year',AGE(CURRENT_DATE,MAKE_DATE(TRIM(m."Year")::INTEGER,1,1)))::INTEGER AS years_difference
FROM pandemic.disease_metrics m
WHERE TRIM(m."Year") ~ '^[0-9]{4}$'
 AND TRIM(m."Year")::INTEGER BETWEEN 1 AND 9999
ORDER BY m.entity_id, source_year
LIMIT 20'''
display(query(sql_task4))

,entity_id,source_year,year_start,today,years_difference
0,1,1971,1971-01-01,2026-10-27,55
1,1,1972,1972-01-01,2026-10-27,54
2,1,1976,1976-01-01,2026-10-27,50
3,1,1980,1980-01-01,2026-10-27,46
4,1,1981,1981-01-01,2026-10-27,45
5,1,1982,1982-01-01,2026-10-27,44
6,1,1983,1983-01-01,2026-10-27,43
7,1,1984,1984-01-01,2026-10-27,42
8,1,1985,1985-01-01,2026-10-27,41
9,1,1986,1986-01-01,2026-10-27,40


## Task 5 — Two independent SQL functions
`years_since(year)` повертає кількість повних років. `cases_per_period(annual_cases, divisor)` захищає від ділення на нуль. Для другої функції використовуємо тільки числові значення.

In [9]:
functions_sql = '''CREATE OR REPLACE FUNCTION pandemic.years_since(p_year INTEGER)
RETURNS INTEGER LANGUAGE SQL STABLE AS $$
 SELECT CASE WHEN p_year BETWEEN 1 AND 9999
 THEN DATE_PART('year', AGE(CURRENT_DATE, MAKE_DATE(p_year,1,1)))::INTEGER
 ELSE NULL END
$$;

CREATE OR REPLACE FUNCTION pandemic.cases_per_period(p_annual_cases NUMERIC, p_divisor NUMERIC)
RETURNS NUMERIC LANGUAGE SQL IMMUTABLE AS $$
 SELECT p_annual_cases / NULLIF(p_divisor,0)
$$;'''
for statement in functions_sql.split(';'):
    if statement.strip(): run(statement)
display(query('SELECT pandemic.years_since(1996) AS years_since_1996, pandemic.cases_per_period(120,12) AS monthly_example'))

,years_since_1996,monthly_example
0,30,10.0


In [10]:
sql_task5 = '''SELECT m.entity_id, m."Year" AS source_year,
 pandemic.years_since(TRIM(m."Year")::INTEGER) AS years_since,
 NULLIF(TRIM(m."Number_rabies"),'')::NUMERIC AS annual_rabies,
 ROUND(pandemic.cases_per_period(NULLIF(TRIM(m."Number_rabies"),'')::NUMERIC,12),3) AS monthly_avg,
 ROUND(pandemic.cases_per_period(NULLIF(TRIM(m."Number_rabies"),'')::NUMERIC,4),3) AS quarterly_avg,
 ROUND(pandemic.cases_per_period(NULLIF(TRIM(m."Number_rabies"),'')::NUMERIC,2),3) AS half_year_avg
FROM pandemic.disease_metrics m
WHERE TRIM(m."Number_rabies") ~ '^[+-]?[0-9]+([.][0-9]+)?$'
 AND TRIM(m."Year") ~ '^[0-9]{4}$'
 AND TRIM(m."Year")::INTEGER BETWEEN 1 AND 9999
ORDER BY m.entity_id, source_year LIMIT 20'''
display(query(sql_task5))

,entity_id,source_year,years_since,annual_rabies,monthly_avg,quarterly_avg,half_year_avg
0,1,1990,36,2.735583,0.228,0.684,1.368
1,1,1991,35,2.712278,0.226,0.678,1.356
2,1,1992,34,2.673644,0.223,0.668,1.337
3,1,1993,33,2.627400,0.219,0.657,1.314
4,1,1994,32,2.575790,0.215,0.644,1.288
5,1,1995,31,2.524039,0.210,0.631,1.262
6,1,1996,30,2.447466,0.204,0.612,1.224
7,1,1997,29,2.337964,0.195,0.584,1.169
8,1,1998,28,2.225475,0.185,0.556,1.113
9,1,1999,27,2.132815,0.178,0.533,1.066


## Final checks
Виконайте після всіх клітинок. Після успіху збережіть notebook із outputs та зробіть скриншоти 1–5 завдань.

In [11]:
counts=query('SELECT (SELECT COUNT(*) FROM pandemic.infectious_cases) AS source_rows,(SELECT COUNT(*) FROM pandemic.disease_metrics) AS normalized_rows,(SELECT COUNT(*) FROM pandemic.entities) AS entities')
display(counts)
r=counts.iloc[0]
assert r.source_rows>0 and r.source_rows==r.normalized_rows and r.entities>0
assert len(query(sql_task3))<=10
assert len(query(sql_task4))>0
assert len(query(sql_task5))>0
assert query('SELECT pandemic.cases_per_period(120,12) AS x').iloc[0,0]==10
assert query('SELECT pandemic.cases_per_period(120,4) AS x').iloc[0,0]==30
assert query('SELECT pandemic.cases_per_period(120,2) AS x').iloc[0,0]==60
print('FINAL CHECK PASSED')

,source_rows,normalized_rows,entities
0,10521,10521,245


FINAL CHECK PASSED
